# Monthly Salary Prediction

## 1. Problem Definition

The objective of this analysis is to predict the midpoint of the monthly
salary range offered in an IT job posting.

The target variable is `salary_mid_pln_monthly`.

The model may help identify which job characteristics are most strongly
associated with salary differences. It is not intended to estimate an
individual candidate's market value.

Only job offers with reliable and comparable salary information are included.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "nofluff_it_jobs_clean.csv"
)

DATA_PATH

WindowsPath('G:/pandas/job_market_intelligence/data/processed/nofluff_it_jobs_clean.csv')

In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 3,360
Columns: 38


In [4]:
model_columns = [
    "salary_mid_pln_monthly",
    "salary_analysis_eligible",
    "experience",
    "experience_years_min",
    "category",
    "contract_type",
    "workplace",
    "job_locations",
    "company_size_segment",
    "required_skills",
]

df[model_columns].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3360 entries, 0 to 3359
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   salary_mid_pln_monthly    2352 non-null   float64
 1   salary_analysis_eligible  3360 non-null   bool   
 2   experience                3332 non-null   object 
 3   experience_years_min      1793 non-null   float64
 4   category                  3360 non-null   object 
 5   contract_type             2589 non-null   object 
 6   workplace                 3070 non-null   object 
 7   job_locations             3360 non-null   object 
 8   company_size_segment      3360 non-null   object 
 9   required_skills           3352 non-null   object 
dtypes: bool(1), float64(2), object(7)
memory usage: 239.7+ KB


In [5]:
df["salary_mid_pln_monthly"].describe().round(2)

count     2352.00
mean     24576.03
std       7510.16
min        100.00
25%      20160.00
50%      24360.00
75%      29030.00
max      81900.00
Name: salary_mid_pln_monthly, dtype: float64

In [8]:
salary_columns = [
    column
    for column in df.columns
    if "salary" in column.lower()
]

salary_columns

['salary_min',
 'salary_max',
 'salary_currency',
 'salary_period',
 'salary_exchange_rate_to_pln',
 'salary_mid',
 'salary_min_pln_monthly',
 'salary_max_pln_monthly',
 'salary_mid_pln_monthly',
 'salary_normalization_eligible',
 'salary_analysis_eligible',
 'salary_exchange_rate_date']

0        True
1        True
2       False
3       False
4        True
        ...  
3355     True
3356     True
3357     True
3358     True
3359     True
Name: salary_analysis_eligible, Length: 3360, dtype: bool

## 2. Modeling Dataset

Only offers marked as eligible for salary analysis are included. Salary
midpoints below PLN 5,000 or above PLN 100,000 were excluded during data
preparation as implausible source-data values.

In [9]:
model_df = df.loc[
    df["salary_analysis_eligible"]
].copy()

print(f"All offers: {len(df):,}")
print(f"Modeling observations: {len(model_df):,}")
print(
    "Excluded observations:",
    len(df) - len(model_df),
)

All offers: 3,360
Modeling observations: 2,339
Excluded observations: 1021


In [10]:
model_df["salary_mid_pln_monthly"].describe().round(2)

count     2339.00
mean     24703.59
std       7331.91
min       5000.00
25%      20160.00
50%      24360.00
75%      29030.00
max      81900.00
Name: salary_mid_pln_monthly, dtype: float64

In [11]:
assert model_df["salary_mid_pln_monthly"].notna().all()

assert model_df[
    "salary_mid_pln_monthly"
].between(5_000, 100_000).all()

In [12]:
target = "salary_mid_pln_monthly"

categorical_features = [
    "experience",
    "category",
    "contract_type",
    "workplace",
    "job_locations",
    "company_size_segment",
]

numeric_features = [
    "experience_years_min",
]

feature_columns = (
    categorical_features
    + numeric_features
)

X = model_df[feature_columns].copy()
y = model_df[target].copy()

In [13]:
feature_quality = pd.DataFrame({
    "dtype": X.dtypes.astype(str),
    "missing": X.isna().sum(),
    "missing_pct": (
        X.isna().mean() * 100
    ).round(1),
    "unique_values": X.nunique(dropna=True),
})

feature_quality

,dtype,missing,missing_pct,unique_values
experience,object,0,0.0,4
category,object,0,0.0,31
contract_type,object,365,15.6,4
workplace,object,150,6.4,2
job_locations,object,0,0.0,17
company_size_segment,object,0,0.0,5
experience_years_min,float64,1035,44.2,11


In [14]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Target missing:", y.isna().sum())

X shape: (2339, 7)
y shape: (2339,)
Target missing: 0


## 3. Train-Test Split

The dataset is divided into an 80% training set and a 20% test set.
The test set is kept separate and will only be used for the final evaluation.

A fixed random state is used to make the experiment reproducible.

In [15]:
from sklearn.model_selection import train_test_split

In [16]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )
)

print("Training observations:", len(X_train))
print("Test observations:", len(X_test))

Training observations: 1871
Test observations: 468


## 4. Baseline Model

The baseline predicts the median salary observed in the training set for every
job offer. More advanced models should produce a lower prediction error than
this simple strategy.

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
)
from sklearn.linear_model import LinearRegression

In [20]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="Missing",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                drop="first",
            ),
        ),
    ]
)

In [21]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
        (
            "scaler",
            StandardScaler(),
        ),
    ]
)

In [22]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_pipeline,
            categorical_features,
        ),
        (
            "numeric",
            numeric_pipeline,
            numeric_features,
        ),
    ]
)

In [23]:
linear_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor,
        ),
        (
            "model",
            LinearRegression(),
        ),
    ]
)

In [24]:
linear_model.fit(
    X_train,
    y_train,
)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categorical', ...), ('numeric', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
linear_predictions = linear_model.predict(
    X_test
)

G:\pandas\.venv\lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [4] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [26]:
linear_mae = mean_absolute_error(
    y_test,
    linear_predictions,
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_predictions,
    )
)

linear_r2 = r2_score(
    y_test,
    linear_predictions,
)

linear_results = pd.DataFrame({
    "model": ["Linear Regression"],
    "MAE": [linear_mae],
    "RMSE": [linear_rmse],
    "R2": [linear_r2],
}).round(2)

linear_results

,model,MAE,RMSE,R2
0,Linear Regression,4414.67,6003.62,0.38
